In [ ]:
# Instalação (executada automaticamente pelo Binder via requirements.txt)
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit import ParameterVector

print("Bibliotecas importadas com sucesso!")

In [ ]:
# 1. Hamiltoniano do modelo de Ising 1D com campo transverso (4 qubits)
n_qubits = 4
J, h = 1.0, 1.0

H = SparsePauliOp.from_list([("I" * n_qubits, 0.0)])  # iniciar com 0
for i in range(n_qubits - 1):
    H += -J * SparsePauliOp.from_list([("I" * i + "ZZ" + "I" * (n_qubits - i - 2), 1.0)])
for i in range(n_qubits):
    H += -h * SparsePauliOp.from_list([("I" * i + "X" + "I" * (n_qubits - i - 1), 1.0)])

print("Hamiltoniano criado:")
print(H)

In [ ]:
# 2. Circuito variacional: 6 camadas de rotações Ry/Rz seguidas de CNOTs
n_layers = 6
n_params = n_qubits * 2 * n_layers
params_vec = ParameterVector('θ', length=n_params)

qc = QuantumCircuit(n_qubits)
idx = 0
for _ in range(n_layers):
    for q in range(n_qubits):
        qc.ry(params_vec[idx], q)
        idx += 1
    for q in range(n_qubits):
        qc.rz(params_vec[idx], q)
        idx += 1
    for q in range(n_qubits - 1):
        qc.cx(q, q + 1)

qc.draw('mpl')

In [ ]:
# 3. Funções auxiliares (todas usam arrays NumPy para evitar depreciação)
backend = Aer.get_backend('statevector_simulator')

def run_circuit_as_array(params):
    """Executa o circuito e retorna o statevector como um array numpy."""
    bound = qc.assign_parameters(params)
    job = backend.run(transpile(bound, backend))
    state = job.result().get_statevector()
    return np.asarray(state)

def energy(params):
    """Energia esperada ⟨ψ|H|ψ⟩ usando a matriz densa de H."""
    psi = run_circuit_as_array(params)
    H_mat = H.to_matrix(sparse=False)  # matriz 16x16 (4 qubits)
    return (psi.conj() @ H_mat @ psi).real

def quantum_fisher_trace(params, eps=1e-6):
    """
    Traço da informação de Fisher quântica (F^Q = 4 * g_FS).
    Calculado via diferenças finitas de segunda ordem nos elementos diagonais
    da métrica de Fubini-Study.
    """
    psi = run_circuit_as_array(params)
    trace = 0.0
    for i in range(len(params)):
        p_plus = params.copy()
        p_plus[i] += eps
        p_minus = params.copy()
        p_minus[i] -= eps
        psi_plus = run_circuit_as_array(p_plus)
        psi_minus = run_circuit_as_array(p_minus)

        overlap_plus = np.abs(np.dot(psi.conj(), psi_plus)) ** 2
        overlap_minus = np.abs(np.dot(psi.conj(), psi_minus)) ** 2

        # Elemento diagonal g_ii da métrica Fubini-Study
        gii = (overlap_plus + overlap_minus - 2) / (4 * eps**2)
        trace += abs(gii)

    return 4.0 * trace  # F^Q = 4 * g_FS

def gradient_energy(params, shift=0.01):
    """Gradiente da energia usando parameter-shift."""
    grad = np.zeros_like(params)
    for i in range(len(params)):
        p_plus = params.copy()
        p_plus[i] += shift
        p_minus = params.copy()
        p_minus[i] -= shift
        E_plus = energy(p_plus)
        E_minus = energy(p_minus)
        grad[i] = (E_plus - E_minus) / (2 * shift)
    return grad

print("Funções definidas.")

In [ ]:
# 4. Algoritmo Geo-SGD quântico
def geo_sgd_quantum(eta=0.1, n_steps=200, epsilon=1e-8):
    np.random.seed(42)
    params = np.random.uniform(0, 2 * np.pi, n_params)
    energies = []
    for step in range(n_steps):
        E_val = energy(params)
        T_Q = quantum_fisher_trace(params)
        grad = gradient_energy(params)

        # Campo geométrico: λ = 2 / √(T_Q + ε)
        lambda_field = 2.0 / np.sqrt(T_Q + epsilon)
        params -= eta * lambda_field * grad
        energies.append(E_val)

        if step % 50 == 0:
            print(f"[Geo-SGD] Passo {step:3d}, E = {E_val:.6f}, T_Q = {T_Q:.4f}, λ = {lambda_field:.4f}")
    return params, energies

print("Geo-SGD definido.")

In [ ]:
# 5. SGD padrão (para comparação)
def sgd_quantum(eta=0.1, n_steps=200):
    np.random.seed(42)
    params = np.random.uniform(0, 2 * np.pi, n_params)
    energies = []
    for step in range(n_steps):
        E_val = energy(params)
        grad = gradient_energy(params)
        params -= eta * grad
        energies.append(E_val)

        if step % 50 == 0:
            print(f"[SGD] Passo {step:3d}, E = {E_val:.6f}")
    return params, energies

print("SGD definido.")

In [ ]:
# 6. Execução e comparação
print("="*50)
print("Executando Geo-SGD ...")
params_geo, eng_geo = geo_sgd_quantum(eta=0.1, n_steps=200)

print("\nExecutando SGD padrão ...")
params_sgd, eng_sgd = sgd_quantum(eta=0.1, n_steps=200)

# Energia exata do estado fundamental (diagonalização exata)
H_mat = H.to_matrix(sparse=False)
E_exact = np.linalg.eigvalsh(H_mat)[0]
print(f"\nEnergia exata do estado fundamental: {E_exact:.6f}")

In [ ]:
# 7. Gráfico comparativo
plt.figure(figsize=(8, 5))
plt.plot(eng_geo, label='Geo-SGD', linewidth=2)
plt.plot(eng_sgd, label='SGD', linewidth=2)
plt.axhline(y=E_exact, color='k', linestyle='--', label='Exata')
plt.xlabel('Passo de otimização')
plt.ylabel('Energia')
plt.title('VQE para o modelo de Ising 1D (4 qubits)')
plt.legend()
plt.grid(True)
plt.show()